### Credit Risk - KNN

1. Import Libraries

In [ ]:
# Basic data handling
import pandas as pd
import numpy as np

# For splitting the dataset
from sklearn.model_selection import train_test_split

# Preprocessing tools
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# KNN Classifier
from sklearn.neighbors import KNeighborsClassifier

# Evaluation metrics
from sklearn.metrics import accuracy_score, confusion_matrix

# To load dataset directly from OpenML
from sklearn.datasets import fetch_openml

2. Load the Dataset

In [ ]:
# Load the German Credit dataset (ID = 31)
data = fetch_openml("credit-g", version=1, as_frame=True)

# Extract the DataFrame
df = data.frame

# Display first few rows
df.head()

3. Check for Missing Values & Drop Them

In [ ]:
# Check how many missing values exist in each column
df.isna().sum()

# Drop rows with ANY missing values
df = df.dropna()

# Confirm no missing values remain
df.isna().sum()

4. Feature Selection (4 numeric + 3 nominal)

In [ ]:
# Select numeric features
numeric_features = ["duration", "credit_amount", "installment_commitment", "age"]

# Select nominal (categorical) features
categorical_features = ["checking_status", "credit_history", "purpose"]

# Target variable
target = "class"

# Create X (features) and y (target)
X = df[numeric_features + categorical_features]
y = df[target]

5. Preprocessing Pipeline

In [ ]:
# Preprocessing for numeric features: Standard Scaling
numeric_transformer = StandardScaler()

# Preprocessing for categorical features: One-Hot Encoding
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

# Combine both preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

6. Split Data (80% train, 10% validation, 10% test)

In [ ]:
# First split: train (80%) + temp (20%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Second split: validation (10%) + test (10%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

7. Train KNN & Tune k Using Validation Set

In [ ]:
# Try different values of k
k_values = range(1, 21)

best_k = None
best_accuracy = 0

for k in k_values:
    # Build pipeline: preprocessing + KNN
    model = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])
    
    # Train model
    model.fit(X_train, y_train)
    
    # Validate model
    y_pred_val = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred_val)
    
    print(f"k = {k}, Validation Accuracy = {acc:.4f}")
    
    # Track best k
    if acc > best_accuracy:
        best_accuracy = acc
        best_k = k

print("\nBest k:", best_k)
print("Best Validation Accuracy:", best_accuracy)

8. Train Final Model Using Best k

In [ ]:
# Build final model with best k
final_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("knn", KNeighborsClassifier(n_neighbors=best_k))
])

# Train on full training set
final_model.fit(X_train, y_train)

9. Evaluate on Test Set

In [ ]:
# Predict on test data
y_pred_test = final_model.predict(X_test)

# Accuracy
test_accuracy = accuracy_score(y_test, y_pred_test)

print("Final Test Accuracy:", test_accuracy)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_test)
print("\nConfusion Matrix:\n", cm)